# Team CatBoost 피처 + 기존 앙상블 개선

팀원 코드에서 하이퍼파라미터는 가져오지 않고, 규정에 맞는 물리 채널차와 동시간 공간 피처만 반영합니다. 최종 권장 실행은 고정된 HM 54피처 CatBoost를 시드 42~44로 확인하고, 각 시드의 2022~2024 rolling OOF 가중치로 기존 HM 모델에 낮은 비율로 결합합니다. TA는 기존 최종 예측을 유지합니다.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    preferred = Path('/content/drive')
    if (preferred / 'MyDrive').exists():
        MOUNT = preferred
    else:
        # stale files가 있는 mountpoint를 피해서 Mountpoint must not already contain files 방지
        MOUNT = preferred if (not preferred.exists() or not any(preferred.iterdir())) else Path('/content/sme_drive')
        MOUNT.mkdir(parents=True, exist_ok=True)
        drive.mount(str(MOUNT), force_remount=False)
    MYDRIVE = MOUNT / 'MyDrive'
else:
    MYDRIVE = Path(os.environ.get('MYDRIVE', '.')).resolve()
print('MYDRIVE =', MYDRIVE)

In [ ]:
REPO_URL = 'https://github.com/tswaincae1221/SME_DATA.git'
BRANCH = 'agent/team-catboost-feature-ensemble'
REPO_DIR = Path('/content/SME_DATA_team_feature_ensemble') if IN_COLAB else Path.cwd()
if IN_COLAB:
    if (REPO_DIR / '.git').exists():
        subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
        subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR, check=True)
        subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    else:
        subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'catboost==1.2.8', 'pandas', 'numpy', 'scikit-learn', 'pytest'], check=True)
print('REPO_DIR =', REPO_DIR)

## 입력 자동 탐색

기존 실험 산출물과 마스터 CSV가 `MyDrive/SME_DATA` 아래에 있다는 전제입니다. 자동 탐색이 여러 파일을 찾으면 아래 경로를 직접 지정하세요.

In [ ]:
DRIVE_ROOT = MYDRIVE / 'SME_DATA'

def first_match(name):
    matches = sorted(DRIVE_ROOT.rglob(name))
    if not matches:
        raise FileNotFoundError(f'{DRIVE_ROOT} 아래에서 {name}을 찾지 못했습니다.')
    print(name, '->', matches[0])
    return matches[0]

MASTER_CSV = first_match('final_train_dataset_19to25_master.csv')
STATION_LIST = first_match('station_list*.csv')
CURRENT_MULTI = first_match('multiseed_report_predictions.csv')

baseline_files = sorted(DRIVE_ROOT.rglob('step0_current_baseline_HM_oof.csv'))
def baseline_for_seed(seed):
    if seed == 42:
        candidates = [p.parent for p in baseline_files if 'seed43' not in str(p).lower() and 'seed44' not in str(p).lower()]
    else:
        candidates = [p.parent for p in baseline_files if f'seed{seed}' in str(p).lower()]
    if not candidates:
        raise FileNotFoundError(f'seed {seed} baseline OOF 디렉터리가 없습니다.')
    return candidates[0]
BASELINE_DIRS = [baseline_for_seed(seed) for seed in [42, 43, 44]]
for seed, path in zip([42, 43, 44], BASELINE_DIRS): print('baseline', seed, '->', path)
OUTPUT_DIR = DRIVE_ROOT / 'processed_station_features/model_experiments_19to25/team_catboost_feature_ensemble_frozen_multiseed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('OUTPUT_DIR =', OUTPUT_DIR)

In [ ]:
# 규정/공간 피처 계약 테스트
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_team_catboost_feature_ensemble.py'], cwd=REPO_DIR, check=True)

## 최종 권장: 고정 피처 다중 시드 확인

피처를 시드마다 다시 고르지 않습니다. 기존 CatBoost 설정도 바꾸지 않습니다. HM 보조 분기의 비율만 각 시드의 2022~2024 rolling OOF에서 선택합니다.

In [ ]:
cmd = [
    sys.executable, '-u', str(REPO_DIR / 'scripts/confirm_team_catboost_feature_ensemble_multiseed.py'),
    '--master-csv', str(MASTER_CSV), '--station-list', str(STATION_LIST),
    '--current-multiseed-predictions', str(CURRENT_MULTI),
    '--seeds', '42', '43', '44', '--baseline-dirs', *map(str, BASELINE_DIRS),
    '--output-dir', str(OUTPUT_DIR), '--threads', '4',
]
print(' '.join(map(str, cmd)))
subprocess.run(cmd, cwd=REPO_DIR, check=True)

In [ ]:
import pandas as pd
summary = json.loads((OUTPUT_DIR / 'frozen_multiseed_summary.json').read_text(encoding='utf-8'))
display(pd.DataFrame([summary['report_metrics']]))
display(pd.read_csv(OUTPUT_DIR / 'frozen_multiseed_metrics.csv'))
print('최종 예측:', OUTPUT_DIR / 'frozen_multiseed_final_2025.csv')

In [ ]:
parts = []
for seed in [42, 43, 44]:
    part = pd.read_csv(OUTPUT_DIR / f'seed_{seed}/frozen_HM_importance.csv')
    part['seed'] = seed
    parts.append(part)
importance = (pd.concat(parts).groupby('feature', as_index=False)
              .agg(mean_importance=('importance', 'mean'), std_importance=('importance', 'std'))
              .sort_values('mean_importance', ascending=False))
display(importance.head(25))

## 선택 실험을 다시 보고 싶을 때

그룹별 발견 실험은 `scripts/experiment_team_catboost_feature_ensemble.py`입니다. 최종 제출 후보에는 위 고정 다중 시드 결과를 우선 사용하세요. `Year`, `Hour`, `Lag1/Roll3`, 검증구간 `bfill`, 실제 TA 기반 HM 힌트는 포함하지 않습니다.